# UPanda Demo: DESI BAO DR2 + emcee

This notebook walks through the full UPanda workflow for a simple BAO analysis:

1. Load a pre-trained JAX emulator for DESI BAO DR2 observables
2. Set up the DESI BAO DR2 likelihood (bundled data, no download needed)
3. Run an MCMC chain with `emcee`
4. Plot the posterior

**Requirements:** `pip install upanda` (or `pip install -e .` from the repo root).  
GPU acceleration is optional — this demo runs fine on CPU.

---

**Cite:** Reeves et al. (2023) [arXiv:2309.03258](https://arxiv.org/abs/2309.03258)

## 1. Imports and backend setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import corner

import UPanda.config as config
from UPanda import utils
from UPanda import run_panda
from UPanda.likelihoods import desi_bao_dr2
from UPanda import integrated_model_jax as IntegratedModel_mod

# Use JAX backend for JIT-compiled predictions
# Set to False to use plain NumPy (slower, but no JAX required)
config.set_jax_enabled(True)

print("UPanda ready. JAX enabled:", config.get_jax_enabled())

## 2. Load the BAO emulator

The example emulator is included in `example_models/` and predicts the DESI BAO
observables (D_V/r_s, D_M/r_s, D_H/r_s) from cosmological parameters.

In [ ]:
import os

# Path to the example emulator — adjust if running from a different directory
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
emu_path = os.path.join(repo_root, "example_models", "bao_dr2_emu")

print(f"Loading emulator from: {emu_path}")

# IntegratedModel is the JAX-compatible emulator wrapper.
# restore() loads the TF weights and converts them to JAX/Flax.
bao_emu = IntegratedModel_mod.IntegratedModel(
    keras_model=None,
    input_scaler=None,
    output_scaler=None,
    parameter_names=None,
)
bao_emu.restore(emu_path)

print("Emulator parameter names:", bao_emu.parameter_names)

## 3. Quick emulator check

Verify the emulator returns sensible predictions at a fiducial cosmology.

In [ ]:
import jax.numpy as jnp

# Fiducial flat ΛCDM cosmology
fid_params = {
    "omega_cdm": jnp.array([0.12]),
    "omega_b":   jnp.array([0.022]),
    "h":         jnp.array([0.68]),
    "n_s":       jnp.array([0.97]),
    "sigma8":    jnp.array([0.82]),
    "m_nu":      jnp.array([0.02]),
    "A_lens":    jnp.array([1.0]),
}

pred = bao_emu.predict(fid_params)
print("Emulator prediction shape:", pred.shape)
print("Predicted BAO observables (fiducial):", pred[0])

## 4. Set up the likelihood

`desi_bao_dr2` loads the measured BAO data vector and covariance from the
bundled data files in `src/UPanda/likelihoods/data/desi_bao_dr2/`.

In [ ]:
likelihood = desi_bao_dr2.desi_bao_dr2(
    emu_model_list=[bao_emu],
)

print("Data vector shape:", likelihood.desi_bao_data.shape)
print("Covariance matrix shape:", likelihood.desi_bao_covariance.shape)

## 5. Define the MCMC configuration

We sample over (omega_cdm, omega_b, h, sigma8) with flat priors.  
n_s and m_nu are held fixed.

In [ ]:
# Parameter names and prior bounds (min, max)
param_names = ["omega_cdm", "omega_b", "h", "sigma8"]
prior_bounds = {
    "omega_cdm": (0.08,    0.20),
    "omega_b":   (0.01875, 0.02625),
    "h":         (0.40,    1.00),
    "sigma8":    (0.30,    1.30),
}

# Fixed parameters (not sampled)
cosmology_params_fixed = [
    {"name": "n_s",    "value": 0.9665},
    {"name": "m_nu",   "value": 0.02},
    {"name": "A_lens", "value": 1.0},
]

# Structured dtype for emcee
dtype = np.dtype([(name, float) for name in param_names])
ndim = len(param_names)
print(f"Sampling {ndim} parameters: {param_names}")

## 6. Define the log-probability function

In [ ]:
from UPanda.likelihoods.utils_likelihood import prepare_params_dict

def log_prior(theta):
    """Flat prior within bounds; -inf outside."""
    for val, name in zip(theta, param_names):
        lo, hi = prior_bounds[name]
        if not (lo < val < hi):
            return -np.inf
    return 0.0


def log_prob(theta):
    """Log-posterior = log-prior + log-likelihood."""
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    # Package as structured array expected by prepare_params_dict
    p = np.array(theta, dtype=[(n, float) for n in param_names]).reshape(1)
    params_dict = prepare_params_dict(
        p, dtype, cosmology_params_fixed, additional_args=None
    )

    # Convert to JAX arrays
    params_jax = {k: jnp.array(v) for k, v in params_dict.items()}

    ll = likelihood.compute_like(params_jax)
    return lp + float(ll[0])


# Quick sanity check at fiducial
theta_fid = [0.12, 0.022, 0.68, 0.82]
print("log_prob at fiducial:", log_prob(theta_fid))

## 7. Run emcee

In [ ]:
import emcee

nwalkers = 32
nsteps   = 2000  # increase for production runs

# Initialise walkers in a small ball around the fiducial
rng = np.random.default_rng(42)
p0 = np.array(theta_fid) + rng.normal(scale=1e-3, size=(nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)

print(f"Running {nsteps} steps with {nwalkers} walkers...")
sampler.run_mcmc(p0, nsteps, progress=True)
print("Done.")

## 8. Inspect convergence

In [ ]:
# Autocorrelation time estimate
try:
    tau = sampler.get_autocorr_time()
    print("Autocorrelation times:", dict(zip(param_names, tau.round(1))))
    burn = int(3 * np.max(tau))
    thin = max(1, int(np.min(tau) / 2))
except emcee.autocorr.AutocorrError:
    print("Chain too short for autocorrelation estimate — using burn=200, thin=1")
    burn, thin = 200, 1

print(f"Using burn-in={burn}, thin={thin}")

flat_samples = sampler.get_chain(discard=burn, thin=thin, flat=True)
print(f"Flat samples shape: {flat_samples.shape}")

In [ ]:
fig, axes = plt.subplots(ndim, figsize=(10, 2 * ndim), sharex=True)
chain = sampler.get_chain()
for i, (ax, name) in enumerate(zip(axes, param_names)):
    ax.plot(chain[:, :, i], alpha=0.3, lw=0.5, color="k")
    ax.set_ylabel(name)
    ax.axvline(burn, color="r", ls="--", lw=1, label="burn-in" if i == 0 else "")
axes[0].legend(fontsize=8)
axes[-1].set_xlabel("Step")
fig.suptitle("Walker traces", y=1.01)
plt.tight_layout()
plt.show()

## 9. Posterior corner plot

In [ ]:
labels = [
    r"$\Omega_\mathrm{cdm} h^2$",
    r"$\Omega_b h^2$",
    r"$h$",
    r"$\sigma_8$",
]

fig = corner.corner(
    flat_samples,
    labels=labels,
    truths=theta_fid,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 11},
)
fig.suptitle("DESI BAO DR2 posterior", y=1.02, fontsize=13)
plt.show()

# Print median ± 1σ for each parameter
print("\nPosterior summary (median ± 1σ):")
for i, name in enumerate(param_names):
    q16, q50, q84 = np.percentile(flat_samples[:, i], [16, 50, 84])
    print(f"  {name:12s}: {q50:.4f} +{q84-q50:.4f} -{q50-q16:.4f}")

---

## Next steps

- **Add more likelihoods:** combine BAO with CMB (`planck_2018_highell_lowell`) or
  supernovae (`pantheon_plus`) by summing log-likelihoods.
- **Use YAML config:** `utils.load_config('configs/config_example_desi_bao.yaml')` reads
  the sampler configuration from a file for reproducible runs.
- **Switch to GPU:** `config.set_jax_enabled(True)` with a CUDA install (see README)
  will JIT-compile the emulator forward pass for much faster sampling.
- **Use NUTS/HMC:** replace `emcee.EnsembleSampler` with `run_panda.run_nuts()` (requires
  JAX backend) for gradient-based sampling with fewer function evaluations.